# 05 — Video Inference & Visualization

This notebook scores arbitrary videos, applies EMA smoothing and hysteresis thresholding, and produces CSV, plot, and annotated MP4 outputs.

In [ ]:
import os, sys, json
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/local disk/MyDrive/anomaly_detection'
sys.path.insert(0, os.path.join(BASE_DIR, 'project_src'))

import torch
import numpy as np
import cv2
from tqdm import tqdm

from training.config import ProjectConfig, get_device
from training.model.scratch_3dcnn import Scratch3DCNN
from training.dataset.transforms import VideoTransform, decode_video_clips
from inference.postprocess import (
    ema_smooth, hysteresis_threshold,
    intervals_to_timestamps, generate_score_csv,
    plot_score_timeline
)

config = ProjectConfig()
device = get_device()
print(f'Device: {device}')

## 1. Load Model & Threshold

In [ ]:
model = Scratch3DCNN().to(device)

best_ckpt = os.path.join(config.CHECKPOINT_DIR, f'best_model_seed{config.SEED}.pt')
checkpoint = torch.load(best_ckpt, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Model loaded from: {best_ckpt}')

# Load calibrated threshold
threshold_path = os.path.join(config.CHECKPOINT_DIR, 'calibrated_threshold.json')
with open(threshold_path, 'r') as f:
    threshold_info = json.load(f)
enter_threshold = threshold_info['threshold']
print(f'Calibrated threshold: {enter_threshold:.4f}')

## 2. Score a Video

In [ ]:
def score_video(video_path, model, transform, device, config):
    """Score all clips from a video and return raw scores + timestamps."""
    clips_data = decode_video_clips(
        video_path,
        target_fps=config.TARGET_FPS,
        clip_length=config.CLIP_LENGTH,
        clip_stride=config.CLIP_STRIDE,
    )
    
    if not clips_data:
        print(f'No clips could be extracted from {video_path}')
        return [], []
    
    all_scores = []
    all_timestamps = []
    
    model.eval()
    with torch.no_grad():
        for clip_frames, start_t, end_t in tqdm(clips_data, desc='Scoring clips'):
            clip_tensor = transform(clip_frames).unsqueeze(0).to(device)  # (1, C, T, H, W)
            logit = model(clip_tensor)
            score = torch.sigmoid(logit).item()
            all_scores.append(score)
            all_timestamps.append((start_t, end_t))
    
    return all_scores, all_timestamps

## 3. Run Inference

In [ ]:
# --- SELECT YOUR VIDEO HERE ---
VIDEO_PATH = os.path.join(config.DATA_ROOT, 'your_video.mp4')  # <-- Change this
# For testing, you can use any video from the dataset:
# VIDEO_PATH = '/content/local_data/Fighting/Fighting001_x264.mp4'

assert os.path.exists(VIDEO_PATH), f'Video not found: {VIDEO_PATH}'
video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print(f'Processing: {video_name}')

# Use inference transform (no crop, full frame)
inference_transform = VideoTransform(mode='inference')

raw_scores, timestamps = score_video(
    VIDEO_PATH, model, inference_transform, device, config
)
print(f'\nTotal clips scored: {len(raw_scores)}')
print(f'Score range: [{min(raw_scores):.4f}, {max(raw_scores):.4f}]')

## 4. Post-Processing

In [ ]:
# EMA smoothing
smoothed_scores = ema_smooth(raw_scores, alpha=config.EMA_ALPHA)

# Hysteresis thresholding
clip_intervals = hysteresis_threshold(
    smoothed_scores,
    enter_threshold=enter_threshold,
    exit_delta=config.EXIT_DELTA,
    run_length=config.RUN_LENGTH,
)

# Convert to timestamps
time_intervals = intervals_to_timestamps(clip_intervals, timestamps)

print(f'Detected {len(time_intervals)} anomaly interval(s):')
for i, (start, end) in enumerate(time_intervals):
    duration = end - start
    print(f'  Interval {i+1}: {start:.1f}s - {end:.1f}s (duration: {duration:.1f}s)')

## 5. Generate Outputs

In [ ]:
output_dir = os.path.join(config.OUTPUT_DIR, video_name)
os.makedirs(output_dir, exist_ok=True)

# 1. Score CSV
csv_path = os.path.join(output_dir, f'{video_name}_scores.csv')
generate_score_csv(raw_scores, smoothed_scores, timestamps, time_intervals, csv_path)
print(f'Score CSV: {csv_path}')

# 2. Score timeline plot
plot_path = os.path.join(output_dir, f'{video_name}_score_plot.png')
plot_score_timeline(
    raw_scores, smoothed_scores, timestamps, time_intervals,
    output_path=plot_path, video_name=video_name
)
print(f'Score plot: {plot_path}')

In [ ]:
def create_annotated_video(video_path, scores, timestamps, intervals, output_path, 
                           threshold, fps=10):
    """Create an annotated MP4 with score overlay and colored border."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'Cannot open video: {video_path}')
        return
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, src_fps, (width, height))
    
    # Build a time->score lookup
    def get_score_at_time(t):
        for i, (start, end) in enumerate(timestamps):
            if start <= t <= end:
                return scores[i]
        return 0.0
    
    def is_in_anomaly_interval(t):
        for start, end in intervals:
            if start <= t <= end:
                return True
        return False
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        current_time = frame_idx / src_fps
        score = get_score_at_time(current_time)
        is_anomaly = is_in_anomaly_interval(current_time)
        
        # Draw colored border
        border_color = (0, 0, 255) if is_anomaly else (0, 255, 0)  # Red/Green in BGR
        border_width = 8
        cv2.rectangle(frame, (0, 0), (width-1, height-1), border_color, border_width)
        
        # Draw score overlay
        score_text = f'Score: {score:.3f}'
        status_text = 'ANOMALY DETECTED' if is_anomaly else 'Normal'
        
        # Background rectangle for text
        cv2.rectangle(frame, (10, 10), (300, 70), (0, 0, 0), -1)
        cv2.putText(frame, score_text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, status_text, (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 
                    (0, 0, 255) if is_anomaly else (0, 255, 0), 2)
        
        out.write(frame)
        frame_idx += 1
    
    cap.release()
    out.release()
    print(f'Annotated video saved: {output_path}')

# Generate annotated video
mp4_path = os.path.join(output_dir, f'{video_name}_annotated.mp4')
create_annotated_video(
    VIDEO_PATH, smoothed_scores, timestamps, time_intervals,
    mp4_path, threshold=enter_threshold
)
print(f'\nAll outputs saved to: {output_dir}')

## 6. Display Results

In [ ]:
from IPython.display import Image, display

if os.path.exists(plot_path):
    display(Image(filename=plot_path))
    
print(f'\nOutputs directory: {output_dir}')
print(f'Files created:')
for f in os.listdir(output_dir):
    size_mb = os.path.getsize(os.path.join(output_dir, f)) / 1e6
    print(f'  {f} ({size_mb:.1f} MB)')

## 7. Deliverable Summary

- **Objective**: Infer anomalies on arbitrary videos.
- **Dependencies**: Model checkpoint and calibrated threshold from training.
- **Outputs**: CSV with scores, Timeline plot of scores, Annotated video with predictions.